In [38]:
import torch
import pandas as pd
import numpy as np
def cov(thetas, sigmas, N, ATT):

    ub = thetas + 1.96 * sigmas/ torch.sqrt(torch.tensor(N))
    lb = thetas - 1.96 * sigmas / torch.sqrt(torch.tensor(N))
    coverage = torch.mean( ( (ATT >= lb) & (ATT <= ub) ).float() , 0 )
    interval_length = torch.mean(ub - lb,0)
    return {"Coverage":coverage,
            "interval_length": interval_length}

In [45]:
Ns = [500, 1000, 2000]
folder = "results_low_rank"
models = ['logistic', 'truncated_logistic', 'truncated_adv']
rows = []
methods =    [
        (1, "OLS"),
        (2, "Linear_old"),
        (3, "LASSO "),
        (4, "RF")]
for n in Ns:
    for model in models:
        file_suffix = f"N:{n}_{model}"
        pred_theta = torch.load(f"{folder}/{file_suffix}_pred_theta_OLS.pt")
        pred_sig = torch.load(f"{folder}/{file_suffix}_pred_sig_OLS.pt")
        ATT = torch.load(f"{folder}/{file_suffix}_ATT_calculations.pt")

        for idx, method_name in methods:
            est = pred_theta[:, idx-1]
            sig = pred_sig[:, idx-1]

            bias = (est - ATT).mean().item()#
            if method_name == "OLS":
                normalizer = 1
            else:
                normalizer = n
            rmse = torch.sqrt(torch.mean((est - ATT) ** 2)).item()
            cov_result = cov(est, sig,normalizer, ATT)
            coverage = cov_result["Coverage"].item()
            int_len = cov_result["interval_length"].item()

            rows.append([ method_name, bias, rmse, coverage, int_len])
        df = pd.DataFrame(rows, columns=["Method", "Bias", "RMSE", "Coverage", "Interval Length"])
        rows = []
        print(f"N:{n}, Model: {model}, ATT: {np.round(float(ATT),4)}")
        print(df.round(4).to_string(index=False))

N:500, Model: logistic, ATT: 4.4352
    Method    Bias   RMSE  Coverage  Interval Length
       OLS -0.0187 0.2333      0.96           1.0257
Linear_old  0.4533 0.5569      0.34           0.6141
    LASSO   0.8207 0.8586      0.21           1.2427
        RF -0.0001 0.2651      0.98           1.1075
N:500, Model: truncated_logistic, ATT: 4.2057
    Method   Bias   RMSE  Coverage  Interval Length
       OLS 0.0089 0.2307      0.98           0.9279
Linear_old 0.0428 0.2496      0.83           0.6248
    LASSO  0.5425 0.5866      0.59           1.2050
        RF 0.0103 0.2279      0.99           1.1423
N:500, Model: truncated_adv, ATT: 4.2261
    Method    Bias   RMSE  Coverage  Interval Length
       OLS -0.0200 0.2400      0.95           0.9278
Linear_old  0.0257 0.2544      0.76           0.6249
    LASSO   0.5315 0.5770      0.63           1.2017
        RF -0.0164 0.2378      0.97           1.1341
N:1000, Model: logistic, ATT: 4.4354
    Method    Bias   RMSE  Coverage  Interval Leng

In [40]:
def numerically_low_rank(m, n, r, eps=.75, decay="geom", *, device=None, dtype=None, seed=None):
    """
    Build X with k=min(m,n) singular values:
      - top r ≈ 1
      - tail ≈ eps * (decay)
    """
    if seed is not None:
        torch.manual_seed(seed)

    k = min(m, n)
    # Orthonormal factors
    U, _ = torch.linalg.qr(torch.randn(m, k, device=device, dtype=dtype), mode="reduced")
    V, _ = torch.linalg.qr(torch.randn(n, k, device=device, dtype=dtype), mode="reduced")

    if r > k:
        raise ValueError(f"r (got {r}) must be ≤ min(m,n)={k}")

    # Tail decay shapes the “numerical” rank
    if k - r > 0:
        if decay == "geom":
            tail = eps * torch.logspace(0, -2, k - r, device=device, dtype=dtype)  # eps → eps*1e-2
        elif decay == "flat":
            tail = eps * torch.ones(k - r, device=device, dtype=dtype)
        elif decay == "power":
            idx = torch.arange(1, k - r + 1, device=device)
            tail = eps * (idx.to(dtype=torch.float32) ** -2).to(dtype=dtype)
        else:
            raise ValueError("decay must be one of {'geom','flat','power'}")
        s = torch.cat([torch.ones(r, device=device, dtype=dtype), tail])
    else:
        s = torch.ones(k, device=device, dtype=dtype)

    S = torch.diag(s)
    X = U @ S @ V.T
    return X.T

In [41]:
x =numerically_low_rank(5,1000,2, decay="power")

In [43]:
svals = torch.linalg.svdvals(x)
print(svals[:10])  # should show a drop after r

tensor([1.0000, 1.0000, 0.7500, 0.1875, 0.0833])


In [35]:
m = 10000
X1 = torch.randn(m, 3)
X11 = X1[:, 0]
Z  = torch.randn(m, 2)
eta = torch.randn(m)
eps_x  = torch.randn(m, 3)
eps_y1 = torch.randn(m)
eps_y2 = torch.randn(m)   # reused for both potentials


Y1 = 2 * (X11 > 0).float() + 1 * X11 + Z @torch.ones(2)

In [37]:
Y1.std()

tensor(2.3438)